## 1. Imports & Device Setup

In [ ]:
import os
import json
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from collections import Counter
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report
from tqdm.notebook import tqdm

# ── Reproducibility ──────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Device: MPS (Apple Silicon) → CUDA → CPU ─────────────────────
if torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
elif torch.cuda.is_available():
    DEVICE = torch.device('cuda')
else:
    DEVICE = torch.device('cpu')

print(f'Using device: {DEVICE}')

# ── Output directory ─────────────────────────────────────────────
os.makedirs('../results', exist_ok=True)

## 2. Load SOLD Dataset

In [ ]:
print('Loading SOLD dataset from HuggingFace...')
sold_train_raw = load_dataset('sinhala-nlp/SOLD', split='train')
sold_test_raw  = load_dataset('sinhala-nlp/SOLD', split='test')

print(f'Train samples : {len(sold_train_raw)}')
print(f'Test  samples : {len(sold_test_raw)}')
print(f'\nColumns : {sold_train_raw.column_names}')
print(f'\nSample record:')
print(sold_train_raw[0])

In [ ]:
train_labels = [item['label'] for item in sold_train_raw]
test_labels  = [item['label'] for item in sold_test_raw]

print('Train label distribution:', Counter(train_labels))
print('Test  label distribution:', Counter(test_labels))

## 3. Vocabulary Building

In [ ]:
# ── Tokenizer: whitespace split (word-level) ──────────────────────
def tokenize(text: str):
    """Simple whitespace tokenizer for Sinhala text."""
    if not isinstance(text, str):
        return []
    return text.strip().split()


# ── Special tokens ────────────────────────────────────────────────
PAD_TOKEN = '<PAD>'
UNK_TOKEN = '<UNK>'
MIN_FREQ  = 2          # ignore words that appear only once
MAX_VOCAB = 20000      # cap vocabulary size


def build_vocab(texts, min_freq=MIN_FREQ, max_vocab=MAX_VOCAB):
    counter = Counter()
    for text in texts:
        counter.update(tokenize(text))

    # Keep only tokens that meet minimum frequency
    vocab_tokens = [tok for tok, cnt in counter.most_common(max_vocab)
                    if cnt >= min_freq]

    # Build index mappings
    word2idx = {PAD_TOKEN: 0, UNK_TOKEN: 1}
    for tok in vocab_tokens:
        word2idx[tok] = len(word2idx)

    idx2word = {v: k for k, v in word2idx.items()}
    return word2idx, idx2word


# Build vocab from training data only (no leakage)
train_texts = sold_train_raw['text']
word2idx, idx2word = build_vocab(train_texts)

VOCAB_SIZE = len(word2idx)
PAD_IDX    = word2idx[PAD_TOKEN]
UNK_IDX    = word2idx[UNK_TOKEN]

print(f'Vocabulary size : {VOCAB_SIZE}')
print(f'PAD index       : {PAD_IDX}')
print(f'UNK index       : {UNK_IDX}')

## 4. Dataset & DataLoader

In [ ]:
def encode_text(text, word2idx, max_len=128):
    """Convert text to index sequence, truncate/pad to max_len."""
    tokens = tokenize(text)[:max_len]
    indices = [word2idx.get(tok, UNK_IDX) for tok in tokens]
    # Pad
    pad_len = max_len - len(indices)
    indices = indices + [PAD_IDX] * pad_len
    return indices


def label_to_int(label):
    """Convert label string to integer."""
    mapping = {'NOT': 0, 'OFF': 1}
    if isinstance(label, int):
        return label
    return mapping.get(str(label).upper(), 0)


class SOLDDataset(Dataset):
    def __init__(self, hf_dataset, word2idx, max_len=128):
        self.texts  = hf_dataset['text']
        self.labels = hf_dataset['label']
        self.word2idx = word2idx
        self.max_len  = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        x = encode_text(self.texts[idx], self.word2idx, self.max_len)
        y = label_to_int(self.labels[idx])
        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)


# Hyperparameters
MAX_LEN    = 128
BATCH_SIZE = 64

train_dataset = SOLDDataset(sold_train_raw, word2idx, MAX_LEN)
test_dataset  = SOLDDataset(sold_test_raw,  word2idx, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f'Train batches : {len(train_loader)}')
print(f'Test  batches : {len(test_loader)}')

## 5. Model Definitions

In [ ]:
class RNNClassifier(nn.Module):
    """
    Generic RNN-based text classifier. Supports LSTM and GRU.

    Architecture:
        Embedding → RNN (bidirectional) → mean+max pool → dropout → Linear

    v2 fair-eval changes vs original:
      - Pooling: mean-only → mean+max  (matches NASClassifier in NB03)
      - Classifier input: hidden*dirs → hidden*dirs*2  (flows from mean+max)
      - Training hyperparameters matched in train_model below
    """
    def __init__(
        self,
        vocab_size,
        embed_dim=128,
        hidden_dim=256,
        num_layers=2,
        num_classes=2,
        rnn_type='LSTM',
        dropout=0.3,
        pad_idx=0,
        bidirectional=True
    ):
        super().__init__()

        self.rnn_type       = rnn_type
        self.hidden_dim     = hidden_dim
        self.num_layers     = num_layers
        self.bidirectional  = bidirectional
        self.num_directions = 2 if bidirectional else 1

        self.embedding = nn.Embedding(
            vocab_size, embed_dim, padding_idx=pad_idx
        )

        rnn_cls = nn.LSTM if rnn_type == 'LSTM' else nn.GRU
        self.rnn = rnn_cls(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=bidirectional
        )

        self.dropout    = nn.Dropout(dropout)
        # mean+max pooling doubles the feature vector — matches NASClassifier
        self.classifier = nn.Linear(
            hidden_dim * self.num_directions * 2, num_classes
        )

    def forward(self, x):
        # x: (batch, seq_len)
        emb = self.dropout(self.embedding(x))   # (batch, seq_len, embed_dim)

        if self.rnn_type == 'LSTM':
            out, _ = self.rnn(emb)              # (batch, seq_len, hidden*dirs)
        else:
            out, _ = self.rnn(emb)

        # Mean + Max pooling — matches NASClassifier in NB03
        # Mean captures average behaviour; max captures peak activations
        mean_pool = out.mean(dim=1)             # (batch, hidden*dirs)
        max_pool, _ = out.max(dim=1)            # (batch, hidden*dirs)
        pooled = torch.cat([mean_pool, max_pool], dim=1)  # (batch, hidden*dirs*2)

        pooled = self.dropout(pooled)
        logits = self.classifier(pooled)        # (batch, num_classes)
        return logits


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# ── Model hyperparameters ─────────────────────────────────────────
MODEL_CONFIG = dict(
    vocab_size    = VOCAB_SIZE,
    embed_dim     = 128,
    hidden_dim    = 256,
    num_layers    = 2,
    num_classes   = 2,
    dropout       = 0.3,
    pad_idx       = PAD_IDX,
    bidirectional = True
)

lstm_model = RNNClassifier(rnn_type='LSTM', **MODEL_CONFIG).to(DEVICE)
gru_model  = RNNClassifier(rnn_type='GRU',  **MODEL_CONFIG).to(DEVICE)

print(f'LSTM parameters : {count_parameters(lstm_model):,}')
print(f'GRU  parameters : {count_parameters(gru_model):,}')
print(f'Classifier input: hidden({256}) * dirs(2) * pool(2) = {256*2*2}')


## 6. Training & Evaluation Functions



In [ ]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    all_preds, all_labels = [], []

    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss   = criterion(logits, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(y.cpu().numpy())

    avg_loss = total_loss / len(loader)
    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds, average='macro')
    return avg_loss, acc, f1


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss   = criterion(logits, y)
            total_loss += loss.item()
            preds = logits.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(y.cpu().numpy())

    avg_loss  = total_loss / len(loader)
    acc       = accuracy_score(all_labels, all_preds)
    f1_macro  = f1_score(all_labels, all_preds, average='macro')
    f1_off    = f1_score(all_labels, all_preds, average='binary', pos_label=1)
    precision = precision_score(all_labels, all_preds, average='macro', zero_division=0)
    recall    = recall_score(all_labels, all_preds, average='macro', zero_division=0)

    return {
        'loss'      : avg_loss,
        'accuracy'  : acc,
        'f1_macro'  : f1_macro,
        'f1_off'    : f1_off,
        'precision' : precision,
        'recall'    : recall,
        'preds'     : all_preds,
        'labels'    : all_labels
    }


def train_model(model, model_name, train_loader, test_loader,
                device, epochs=30, lr=3e-4):
    """
    Full training loop with learning rate scheduling.
    Returns best test metrics.

    v2 fair-eval: lr=3e-4, weight_decay=1e-4, scheduler patience=3
    — all identical to NB03 NAS training.
    """
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=3
    )

    best_f1    = 0.0
    best_metrics = {}
    history    = []

    print(f'\n{"="*55}')
    print(f'  Training {model_name}')
    print(f'  lr={lr} | wd=1e-4 | clip=1.0 | mean+max pool | patience=3')
    print(f'{"="*55}')

    for epoch in range(1, epochs + 1):
        train_loss, train_acc, train_f1 = train_epoch(
            model, train_loader, optimizer, criterion, device
        )
        test_metrics = evaluate(model, test_loader, criterion, device)
        scheduler.step(test_metrics['f1_macro'])

        history.append({
            'epoch'      : epoch,
            'train_loss' : train_loss,
            'train_acc'  : train_acc,
            'train_f1'   : train_f1,
            'test_loss'  : test_metrics['loss'],
            'test_acc'   : test_metrics['accuracy'],
            'test_f1'    : test_metrics['f1_macro'],
        })

        print(
            f'Epoch {epoch:02d}/{epochs} | '
            f'Train Loss: {train_loss:.4f} | '
            f'Train F1: {train_f1:.4f} | '
            f'Test Loss: {test_metrics["loss"]:.4f} | '
            f'Test F1: {test_metrics["f1_macro"]:.4f} | '
            f'Test Acc: {test_metrics["accuracy"]:.4f}'
        )

        # Save best model based on test F1
        if test_metrics['f1_macro'] > best_f1:
            best_f1 = test_metrics['f1_macro']
            best_metrics = test_metrics.copy()
            torch.save(model.state_dict(),
                       f'../models/{model_name}_best.pt')
            print(f'  ✓ New best F1: {best_f1:.4f} — model saved')

    return best_metrics, history

## 7. Train LSTM Baseline


In [ ]:
os.makedirs('../models', exist_ok=True)

EPOCHS = 30
LR     = 3e-4   # v2: matched to NAS (was 1e-3)

lstm_model = RNNClassifier(rnn_type='LSTM', **MODEL_CONFIG).to(DEVICE)

lstm_best, lstm_history = train_model(
    lstm_model, 'LSTM',
    train_loader, test_loader,
    DEVICE, epochs=EPOCHS, lr=LR
)

## 8. Train GRU Baseline

In [ ]:
gru_model = RNNClassifier(rnn_type='GRU', **MODEL_CONFIG).to(DEVICE)

gru_best, gru_history = train_model(
    gru_model, 'GRU',
    train_loader, test_loader,
    DEVICE, epochs=EPOCHS, lr=LR
)

## 9. Results Summary

In [ ]:
print('\n' + '='*55)
print('  FINAL RESULTS — SOLD Test Set')
print('='*55)

for name, metrics in [('LSTM', lstm_best), ('GRU', gru_best)]:
    print(f'\n{name}:')
    print(f'  Accuracy        : {metrics["accuracy"]:.4f}')
    print(f'  F1 (Macro)      : {metrics["f1_macro"]:.4f}')
    print(f'  F1 (Offensive)  : {metrics["f1_off"]:.4f}')
    print(f'  Precision (Mac) : {metrics["precision"]:.4f}')
    print(f'  Recall (Macro)  : {metrics["recall"]:.4f}')

print()

# Detailed classification report
print('\n── LSTM Classification Report ──')
print(classification_report(
    lstm_best['labels'], lstm_best['preds'],
    target_names=['NOT', 'OFF']
))

print('\n── GRU Classification Report ──')
print(classification_report(
    gru_best['labels'], gru_best['preds'],
    target_names=['NOT', 'OFF']
))

## 10. Save Results & Vocabulary

In [ ]:
# Save baseline results
results = {
    'LSTM': {
        'accuracy'  : lstm_best['accuracy'],
        'f1_macro'  : lstm_best['f1_macro'],
        'f1_off'    : lstm_best['f1_off'],
        'precision' : lstm_best['precision'],
        'recall'    : lstm_best['recall'],
        'history'   : lstm_history
    },
    'GRU': {
        'accuracy'  : gru_best['accuracy'],
        'f1_macro'  : gru_best['f1_macro'],
        'f1_off'    : gru_best['f1_off'],
        'precision' : gru_best['precision'],
        'recall'    : gru_best['recall'],
        'history'   : gru_history
    },
    'config': {
        'vocab_size'  : VOCAB_SIZE,
        'max_len'     : MAX_LEN,
        'batch_size'  : BATCH_SIZE,
        'epochs'      : EPOCHS,
        'lr'          : LR,
        'weight_decay' : 1e-4,
        'pooling'      : 'mean+max',
        'scheduler_patience': 3,
        'version'      : 'v2_fair_eval',
        'model_config': MODEL_CONFIG
    }
}

with open('../results/baseline_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print('Baseline results saved to ../results/baseline_results.json')

# Save vocabulary for use in notebook 02 and 03
with open('../results/vocab.json', 'w', encoding='utf-8') as f:
    json.dump(word2idx, f, ensure_ascii=False, indent=2)

print('Vocabulary saved to ../results/vocab.json')
print(f'\nVocab size: {VOCAB_SIZE}')

## 11. Quick Training Curve Plot

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, history, name in zip(axes,
                              [lstm_history, gru_history],
                              ['LSTM', 'GRU']):
    epochs_list  = [h['epoch']    for h in history]
    train_f1_list = [h['train_f1'] for h in history]
    test_f1_list  = [h['test_f1']  for h in history]

    ax.plot(epochs_list, train_f1_list, label='Train F1', marker='o')
    ax.plot(epochs_list, test_f1_list,  label='Test F1',  marker='s')
    ax.set_title(f'{name} — F1 over Epochs')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Macro F1')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/baseline_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved to ../results/baseline_training_curves.png')